# Tutorial 1 — The Vanilla LLM Backend & Instruction Design

Before agents, before tool use, before ontology — there is one primitive: `backend.complete(system, user)`. It takes a system prompt, a user message, and returns text. Everything else SEOCHO does is built on top of this call.

This tutorial covers:

1. The provider registry — OpenAI / xAI Grok / Moonshot Kimi / DeepSeek / Alibaba Qwen, all behind one interface.
2. Your first `complete()` call.
3. **Instruction design** — the part of the API you spend the most time tuning. The system prompt is what makes the same model behave like five different tools.
4. Provider quirks SEOCHO handles for you (Kimi temperature clamp, JSON-mode fallback, reasoning-content fallback).
5. Where this sits in the rest of SEOCHO (pointer to Tutorial 2 for agents, Tutorial 3 for ontology).

Source references — `seocho/store/llm.py` (`OpenAICompatibleBackend`, `create_llm_backend`, `_PROVIDER_SPECS`).

## 0. Setup — run me first


In [ ]:
# --- Setup (run me first) ---
import sys
from pathlib import Path

try:
    import seocho
    from dotenv import load_dotenv
except ImportError:
    %pip install -q seocho python-dotenv
    import seocho
    from dotenv import load_dotenv

load_dotenv(Path.cwd().parent / ".env", override=False)
print('seocho', seocho.__version__, '| .env loaded')


## 1. Provider registry

In [25]:
from seocho import list_provider_specs

specs = list_provider_specs()
print(f'{"provider":<10} {"env var":<22} {"default model":<28} base_url')
print('-' * 100)
for name, spec in specs.items():
    print(f'{name:<10} {spec.api_key_env:<22} {spec.default_model:<28} {spec.base_url or "(OpenAI default)"}')

provider   env var                default model                base_url
----------------------------------------------------------------------------------------------------
openai     OPENAI_API_KEY         gpt-4o                       (OpenAI default)
deepseek   DEEPSEEK_API_KEY       deepseek-chat                https://api.deepseek.com
kimi       MOONSHOT_API_KEY       kimi-k2.5                    https://api.moonshot.ai/v1
grok       XAI_API_KEY            grok-4.20-reasoning          https://api.x.ai/v1
qwen       DASHSCOPE_API_KEY      qwen-plus                    https://dashscope-intl.aliyuncs.com/compatible-mode/v1


Each row is a `ProviderSpec` (`seocho/store/llm.py:20`). `OpenAICompatibleBackend.__init__` resolves the API key via `os.getenv(spec.api_key_env)` and the base URL from `spec.base_url`. Once an env var is set, instantiation is one line.

## 2. Your first `complete()` call

In [26]:
import os, getpass

PROVIDERS = [
    ('openai',   'OPENAI_API_KEY',     'gpt-4o-mini'),
    ('grok',     'XAI_API_KEY',        'grok-4.20-reasoning'),
    ('kimi',     'MOONSHOT_API_KEY',   'kimi-k2.5'),
    ('deepseek', 'DEEPSEEK_API_KEY',   'deepseek-chat'),
]

# Prompt for any keys not already set (from .env, shell, or earlier cells).
# Press Enter to skip a provider you don't have a key for.
for _, env_name, _ in PROVIDERS:
    if not os.environ.get(env_name):
        val = getpass.getpass(f'{env_name} (blank to skip): ')
        if val:
            os.environ[env_name] = val

AVAILABLE = [(p, env, m) for p, env, m in PROVIDERS if os.environ.get(env)]
print('available providers:', [p for p, _, _ in AVAILABLE] or '(none — set at least one *_API_KEY)')

available providers: ['openai', 'grok', 'kimi', 'deepseek']


In [7]:
from seocho import create_llm_backend

if AVAILABLE:
    provider, env_name, model = AVAILABLE[0]
    backend = create_llm_backend(provider=provider, model=model)
    response = backend.complete(
        system='You are a helpful assistant. Answer in one short sentence.',
        user='What is the capital of France?',
    )
    print(f'provider : {provider}')
    print(f'model    : {response.model or model}')
    print(f'usage    : {response.usage}')
    print(f'text     : {response.text.strip()}')
else:
    print('[SKIP] no provider keys set — see section 1 for the env vars.')

[SKIP] no provider keys set — see section 1 for the env vars.


## 3. Same call, every provider

Every provider speaks the OpenAI Chat Completions wire format. The exact same call works against each — just change the `provider` argument. SEOCHO subclasses (`OpenAIBackend`, `KimiBackend`, `GrokBackend`, …) are convenience aliases for `OpenAICompatibleBackend(provider=...)`.

In [27]:
USER = 'In one short sentence, who is the CEO of Apple as of 2025?'

for provider, env_name, model in PROVIDERS:
    if not os.environ.get(env_name):
        print(f'[SKIP] {provider:<10} {env_name} not set')
        continue
    backend = create_llm_backend(provider=provider, model=model)
    try:
        resp = backend.complete(system='You answer in one short sentence.', user=USER)
        text = resp.text.strip().replace(chr(10), ' ')
        print(f'{provider:<10} {text[:140]}')
    except Exception as exc:
        print(f'{provider:<10} ERROR: {type(exc).__name__}: {str(exc)[:120]}')

openai     I cannot provide information about events or changes that occurred after October 2023.
grok       Tim Cook is the CEO of Apple as of 2025.  \confidence{70}
kimi       Tim Cook is the CEO of Apple as of 2025.
deepseek   As of 2025, Tim Cook is the CEO of Apple.


## 4. Instruction design — making one LLM behave many ways

The system prompt is the largest lever you have over output quality. Three patterns that consistently move quality:

1. **Role + scope** — name the role and explicitly close off everything outside it.
2. **Format constraint** — say exactly what shape the output should take. Pair with `response_format={'type': 'json_object'}` for hard JSON.
3. **Few-shot** — paste 1–3 input/output examples in the system prompt. Cheap; large effect on consistency.

All three are pure system-prompt edits — no SEOCHO API change required.

In [28]:
SAMPLE = (
    'Tim Cook works at Apple in Cupertino. He has been CEO since 2011. '
    'Apple announced the new iPhone at WWDC 2025.'
)

# --- Pattern 1: role + scope ----------------------------------
sys_role_scope = (
    'You extract company names ONLY from the input. '
    'If the input has no company, return an empty list. '
    'Reply with a single line of comma-separated company names, nothing else.'
)

# --- Pattern 2: format constraint with JSON mode ---------------
sys_json = (
    'Extract every (person, organization) pair from the input. '
    'Return JSON: {"pairs": [{"person": str, "organization": str}]}'
)

# --- Pattern 3: few-shot in the system prompt -----------------
sys_fewshot = '''Extract executive titles. Return JSON: {"name": str, "title": str, "organization": str}.

Example:
  Input: "Sundar Pichai is the CEO of Google."
  Output: {"name": "Sundar Pichai", "title": "CEO", "organization": "Google"}

Example:
  Input: "Lisa Su leads AMD as president and CEO."
  Output: {"name": "Lisa Su", "title": "CEO", "organization": "AMD"}
'''

if AVAILABLE:
    backend = create_llm_backend(provider=AVAILABLE[0][0], model=AVAILABLE[0][2])
    for label, system, json_mode in [
        ('role+scope', sys_role_scope, False),
        ('json-mode ', sys_json,        True),
        ('few-shot  ', sys_fewshot,    True),
    ]:
        resp = backend.complete(
            system=system,
            user=SAMPLE,
            temperature=0.0,
            response_format={'type': 'json_object'} if json_mode else None,
        )
        text = resp.text.strip().replace(chr(10), ' ')
        print(f'[{label}] {text[:200]}')
else:
    print('[SKIP] instruction-design demo — set at least one provider key.')

[role+scope] Apple
[json-mode ] {   "pairs": [     {       "person": "Tim Cook",       "organization": "Apple"     }   ] }
[few-shot  ] {"name": "Tim Cook", "title": "CEO", "organization": "Apple"}


Notice the contrast: the role+scope prompt produces a one-line list; the JSON-mode prompt produces a dict; the few-shot prompt locks the schema even tighter. Same backend, same model — different system prompt, different tool.

This is the reason ontology-driven systems (Tutorial 3) are powerful: the **ontology is a structured way to generate this system prompt automatically**. `ontology.to_extraction_context()` returns the entity types, relationships, and constraints as text ready to slot into a system prompt. You will see exactly that pattern in Tutorial 3.

## 5. Provider quirks SEOCHO handles for you

Three quirks live inside `OpenAICompatibleBackend` so feature code never branches on the provider name:

**Kimi K2.5 temperature clamp** — `_safe_temperature` (line 236) forces `temperature=1.0` whenever `provider == 'kimi'` because the upstream API rejects any other value. Your code keeps passing `0.0`.

**`reasoning_content` fallback** — `_build_response` (line 406) reads `choice.message.content`, and if that is empty (typical for reasoning-cut Kimi responses) it falls back to `choice.message.reasoning_content`. `LLMResponse.text` is uniformly populated.

**`response_format` graceful retry** — `complete()` (line 269) tries with the JSON-object format and, on any exception when one was passed, drops the kwarg and re-runs with `Return ONLY valid JSON.` appended to the system prompt. Providers without JSON-mode support keep working.

In [31]:
import inspect
from seocho import OpenAICompatibleBackend

print(inspect.getsource(OpenAICompatibleBackend._safe_temperature))

    def _safe_temperature(self, temperature: float) -> float:
        """Clamp temperature for providers with restrictions.

        Kimi K2.5 only accepts temperature=1.  Rather than patching
        every call-site, we enforce the constraint here.
        """
        if self.provider == "kimi":
            return 1.0
        return temperature



In [43]:
help(OpenAICompatibleBackend)

Help on class OpenAICompatibleBackend in module seocho.store.llm:

class OpenAICompatibleBackend(LLMBackend)
 |  OpenAICompatibleBackend(*, provider: 'str' = 'openai', model: 'Optional[str]' = None, api_key: 'Optional[str]' = None, base_url: 'Optional[str]' = None, timeout: 'float' = 120.0) -> 'None'
 |  
 |  LLM backend for OpenAI-compatible chat-completions APIs.
 |  
 |  Method resolution order:
 |      OpenAICompatibleBackend
 |      LLMBackend
 |      abc.ABC
 |      builtins.object
 |  
 |  Methods defined here:
 |  
 |  __init__(self, *, provider: 'str' = 'openai', model: 'Optional[str]' = None, api_key: 'Optional[str]' = None, base_url: 'Optional[str]' = None, timeout: 'float' = 120.0) -> 'None'
 |      Initialize self.  See help(type(self)) for accurate signature.
 |  
 |  __repr__(self) -> 'str'
 |      Return repr(self).
 |  
 |  async acomplete(self, *, system: 'str', user: 'str', temperature: 'float' = 0.0, max_tokens: 'Optional[int]' = None, response_format: 'Optional[Dic

## 6. Provider auto-detection

If you only have a model identifier, `Workbench._resolve_llm` (`seocho/experiment.py:580`) infers the provider from keywords. Easy to inline:

In [44]:
def resolve(model: str) -> str:
    m = model.lower()
    if 'kimi' in m or 'moonshot' in m: return 'kimi'
    if 'deepseek' in m:                return 'deepseek'
    if 'grok' in m:                    return 'grok'
    if 'qwen' in m or 'dashscope' in m: return 'qwen'
    return 'openai'

for model in ['gpt-4o-mini', 'deepseek-chat', 'kimi-k2.5', 'grok-4.20-reasoning', 'qwen-plus']:
    print(f'{model:<24} -> {resolve(model)}')

gpt-4o-mini              -> openai
deepseek-chat            -> deepseek
kimi-k2.5                -> kimi
grok-4.20-reasoning      -> grok
qwen-plus                -> qwen


## 7. Where this sits in the rest of SEOCHO

The vanilla LLM is the floor. Two things you can layer on top:

- **Tutorial 2 — Agent enhancement.** Wraps `complete()` with tool use, multi-turn state, and Opik observability. The reader goes from one-shot prompts to stateful agent conversations.
- **Tutorial 3 — Ontology integration.** Replaces hand-written system prompts with a generated one driven by an ontology contract (FIBO). Same provider, same agent — but now the system prompt is structured and version-checked.

If you only need one-shot extraction or simple chat completions, you can stop here. The `complete()` primitive is enough.